In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date

#read in bronze 

bronze = spark.read.table("dataengineering.v1.bronze")
bronze.printSchema()

In [0]:

from pyspark.sql.functions import regexp_replace
from pyspark.sql import functions as F
# Read fresh data from bronze table
bronze = spark.read.table("dataengineering.v1.bronze")

# Standardize date format by replacing periods with slashes
bronze = bronze.withColumn("signup_date", regexp_replace("signup_date", "\\.", "/"))

# Convert to proper date type
bronze = bronze.withColumn("signup_date", to_date("signup_date", "M/d/yy"))


# Remove duplicates
bronze_cleaned = bronze.dropDuplicates(["user_id"])




In [0]:
bronze_cleaned = bronze_cleaned.withColumn(
    "referral_source", F.lower("referral_source")

)

In [0]:
write_table = "dataengineering.v1.silver"

# Drop the incorrectly named column (typo in Cell 6)
# bronze_cleaned = bronze_cleaned.drop("refeerral_source")

# Write to table
(
    bronze_cleaned
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(write_table)
)

In [0]:
#Gold table 1 - monthly sign ups
gold = spark.read.table("dataengineering.v1.silver")

gold_monthly = gold.groupBy(F.date_format("signup_date", "yyyy-MM").alias("signup_month")).count()

write_table = "dataengineering.v1.gold_monthly"

(
    gold_monthly
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(write_table))
    

In [0]:
# Gold Table  - Signups by country
gold_country = gold.groupBy("country").count()

write_table = "dataengineering.v1.gold_country"

(
    gold_country
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(write_table))
#

In [0]:
#gold table = referral performance

gold_referral = gold.groupBy("referral_source").count()

write_table = "dataengineering.v1.gold_referral"

(
    gold_referral
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(write_table))



In [0]:
#gold table 4- user dim table
gold_users = gold.select(
    "user_id",
    "first_name",
    "last_name",
    "email",
    "signup_date",
    "country",
    "referral_source"

)

write_table = "dataengineering.v1.gold_users"

(
    gold_users
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(write_table))